# 🤖 Lab 2: AI는 어떻게 학습하는가? (30분)

## 학습 목표
1. **학습 전 vs 학습 후** AI 행동의 극적인 차이를 확인한다
2. GA로 신경망을 학습시키는 전체 과정을 체험한다
3. AI 학습 방법들의 **장단점**을 이해한다
4. 자율주행 시뮬레이터 실습을 위한 배경지식을 쌓는다

## 핵심 질문
> "랜덤 숫자 뭉치(신경망)가 어떻게 '지능적 행동'을 하게 되는가?"


## 1단계: 문제 설정 — 장애물 피하기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(42)

# ============================================
# 환경: 간단한 장애물 피하기
# ============================================
# 시작(0, 5) → 목표(10, 5)
# 중간에 벽이 있고, 위쪽에 틈이 있음
# 에이전트는 틈을 찾아서 통과해야 함

class NavEnv:
    """
    자율주행 시뮬레이터의 축소판!
    - 센서 4개: 전방 벽 거리, 위쪽 공간, 아래쪽 공간, 목표 방향
    - 출력 2개: x이동, y이동
    """
    def __init__(self):
        self.start = (0.0, 5.0)
        self.goal = (10.0, 5.0)
        # 벽: x=5 위치, y=0~7 (y>7에 틈)
        self.wall_x = 5.0
        self.wall_gap_y = 7.0  # 이 위로는 뚫려있음
    
    def is_wall(self, x, y):
        if x < 0 or x > 10 or y < 0 or y > 10:
            return True
        if 4.8 < x < 5.2 and y < self.wall_gap_y:
            return True
        return False
    
    def get_sensors(self, x, y):
        # 전방 벽까지 거리 (정규화)
        front_dist = 1.0
        for d in np.arange(0.1, 5.0, 0.1):
            if self.is_wall(x + d, y):
                front_dist = d / 5.0
                break
        
        # 위쪽/아래쪽 공간
        up_space = y / 10.0
        down_space = (10.0 - y) / 10.0
        
        # 목표 방향
        goal_dir = (self.goal[0] - x) / 10.0
        
        return np.array([front_dist, up_space, down_space, goal_dir])
    
    def simulate(self, brain, max_steps=150):
        x, y = self.start
        path = [(x, y)]
        
        for _ in range(max_steps):
            sensors = self.get_sensors(x, y)
            output = brain.forward(sensors)
            
            dx = np.tanh(output[0]) * 0.15
            dy = np.tanh(output[1]) * 0.15
            
            nx = x + dx
            ny = np.clip(y + dy, 0.1, 9.9)
            
            if not self.is_wall(nx, ny):
                x, y = nx, ny
            
            path.append((x, y))
            
            # 목표 도달
            if abs(x - self.goal[0]) < 0.5 and abs(y - self.goal[1]) < 2:
                break
        
        return path
    
    def evaluate(self, brain):
        path = self.simulate(brain)
        final_x, final_y = path[-1]
        
        # 적합도: x 진행도 + 목표 근접도
        x_progress = final_x  # 얼마나 오른쪽으로 갔나 (max 10)
        goal_dist = np.sqrt((final_x - self.goal[0])**2 + (final_y - self.goal[1])**2)
        goal_bonus = max(0, 15 - goal_dist)
        
        return x_progress + goal_bonus  # max ~25
    
    def draw(self, ax, paths=None, title=""):
        ax.set_xlim(-0.5, 10.5)
        ax.set_ylim(-0.5, 10.5)
        ax.set_aspect('equal')
        
        # 벽
        ax.add_patch(patches.Rectangle((4.8, 0), 0.4, self.wall_gap_y, 
                     facecolor='#333', edgecolor='black'))
        
        # 틈 표시
        ax.annotate('Gap', xy=(5, self.wall_gap_y + 0.5), fontsize=9,
                    color='green', ha='center', fontweight='bold')
        
        # 시작/목표
        ax.plot(*self.start, 'go', markersize=12, label='Start', zorder=5)
        ax.plot(*self.goal, 'r*', markersize=15, label='Goal', zorder=5)
        
        if paths:
            colors = plt.cm.tab10(np.linspace(0, 1, len(paths)))
            for i, path in enumerate(paths):
                xs, ys = zip(*path)
                ax.plot(xs, ys, '-', color=colors[i], alpha=0.6, linewidth=1.5)
                ax.plot(xs[-1], ys[-1], 'x', color=colors[i], markersize=8)
        
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.grid(True, alpha=0.2)

env = NavEnv()
print("✅ 환경 생성 완료!")
print(f"   시작: {env.start} → 목표: {env.goal}")
print(f"   벽: x=5 위치, y<{env.wall_gap_y} 구간이 막혀있음")
print(f"   에이전트는 위쪽 틈(y>{env.wall_gap_y})을 찾아서 통과해야 합니다")


In [ ]:
# ============================================
# 신경망 = 에이전트의 "두뇌"
# ============================================
class Brain:
    """
    자율주행 시뮬레이터의 CarBrain과 동일한 구조!
    입력(4) → 은닉층(8, ReLU) → 출력(2)
    총 파라미터: 4×8 + 8 + 8×2 + 2 = 58개
    """
    def __init__(self):
        self.w1 = np.random.randn(4, 8) * np.sqrt(2.0 / 4)
        self.b1 = np.zeros(8)
        self.w2 = np.random.randn(8, 2) * np.sqrt(2.0 / 8)
        self.b2 = np.zeros(2)
    
    def forward(self, x):
        h = np.maximum(0, x @ self.w1 + self.b1)  # ReLU
        return h @ self.w2 + self.b2
    
    def copy(self):
        b = Brain.__new__(Brain)
        b.w1, b.b1 = self.w1.copy(), self.b1.copy()
        b.w2, b.b2 = self.w2.copy(), self.b2.copy()
        return b
    
    def mutate(self, rate=0.2, strength=0.3):
        b = self.copy()
        for attr in ['w1', 'b1', 'w2', 'b2']:
            w = getattr(b, attr)
            mask = np.random.rand(*w.shape) < rate
            w += np.random.randn(*w.shape) * strength * mask
        return b
    
    def crossover(self, other):
        b = Brain.__new__(Brain)
        for attr in ['w1', 'b1', 'w2', 'b2']:
            a_val, b_val = getattr(self, attr), getattr(other, attr)
            mask = np.random.rand(*a_val.shape) < 0.5
            setattr(b, attr, np.where(mask, a_val, b_val))
        return b

brain = Brain()
print(f"✅ 신경망 생성!")
print(f"   구조: 4(센서) → 8(은닉) → 2(이동)")
print(f"   파라미터 58개 = 이 숫자들이 행동 규칙을 결정")
print(f"   지금은 전부 랜덤 → 의미 없는 행동을 할 것입니다")


## 2단계: 학습 전 — 랜덤 두뇌의 행동 관찰

신경망의 58개 파라미터가 **완전히 랜덤**인 상태입니다.

이 상태에서 에이전트가 어떻게 움직이는지 봅시다.


In [ ]:
# ============================================
# 학습 전: 랜덤 두뇌 10개의 행동
# ============================================
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

random_paths = []
random_scores = []

for i in range(10):
    brain = Brain()
    path = env.simulate(brain)
    score = env.evaluate(brain)
    random_paths.append(path)
    random_scores.append(score)

env.draw(ax, random_paths, "BEFORE Training: Random Brains")
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"\n📊 학습 전 결과 (랜덤 두뇌 10개):")
print(f"   평균 적합도: {np.mean(random_scores):.1f} / 25.0")
print(f"   최고 적합도: {np.max(random_scores):.1f}")
print(f"   목표 도달: 0/10")
print(f"\n→ 대부분 벽 앞에서 멈추거나 엉뚱한 방향으로 갑니다")
print("→ 랜덤 숫자로는 '벽 위의 틈을 찾아서 통과'하는 행동이 나올 수 없습니다")


## 3단계: GA로 학습! 🧬

Lab 1에서 배운 유전 알고리즘을 **신경망의 가중치**에 적용합니다.

- 개체 = 신경망 하나 (58개 가중치가 "유전자")
- 적합도 = 얼마나 목표에 가까이 갔는지
- 30개 에이전트가 동시에 시도 → 좋은 놈 선택 → 교차+변이 → 반복


In [ ]:
# ============================================
# GA 학습 실행
# ============================================
POP_SIZE = 30
GENERATIONS = 60

np.random.seed(42)
population = [Brain() for _ in range(POP_SIZE)]

best_history = []
avg_history = []
best_brain = None
best_score = 0

print("=" * 60)
print("🧬 GA 학습 시작!")
print(f"   인구: {POP_SIZE} | 세대: {GENERATIONS}")
print("=" * 60)

for gen in range(GENERATIONS):
    # 평가
    scores = [env.evaluate(b) for b in population]
    
    gen_best = max(scores)
    gen_avg = np.mean(scores)
    best_history.append(gen_best)
    avg_history.append(gen_avg)
    
    # 최고 기록 갱신
    best_idx = np.argmax(scores)
    if scores[best_idx] > best_score:
        best_score = scores[best_idx]
        best_brain = population[best_idx].copy()
    
    if gen % 10 == 0 or gen == GENERATIONS - 1:
        print(f"  세대 {gen+1:3d} | 최고: {gen_best:5.1f} | 평균: {gen_avg:5.1f}")
    
    # 다음 세대
    sorted_pop = sorted(zip(scores, population), key=lambda x: x[0], reverse=True)
    new_pop = [sorted_pop[i][1].copy() for i in range(6)]  # 엘리트 보존
    
    while len(new_pop) < POP_SIZE:
        cands_a = np.random.choice(POP_SIZE, 3, replace=False)
        cands_b = np.random.choice(POP_SIZE, 3, replace=False)
        parent_a = population[cands_a[np.argmax([scores[c] for c in cands_a])]]
        parent_b = population[cands_b[np.argmax([scores[c] for c in cands_b])]]
        child = parent_a.crossover(parent_b).mutate(rate=0.2, strength=0.3)
        new_pop.append(child)
    
    population = new_pop

print(f"\n✅ 학습 완료! 최고 적합도: {best_score:.1f}")


## 4단계: 학습 전 vs 학습 후 비교 👀

In [ ]:
# ============================================
# 학습 전 vs 후 비교!
# ============================================
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 학습 전
before_paths = []
for _ in range(8):
    brain = Brain()
    before_paths.append(env.simulate(brain))
env.draw(axes[0], before_paths, "BEFORE Training (Random)")

# 학습 후
after_paths = []
trained_scores = []
for i in range(8):
    if i == 0:
        brain = best_brain.copy()
    else:
        brain = best_brain.mutate(rate=0.03, strength=0.05)
    path = env.simulate(brain)
    after_paths.append(path)
    trained_scores.append(env.evaluate(brain))
env.draw(axes[1], after_paths, "AFTER Training (Evolved)")

# 학습 곡선
axes[2].plot(best_history, 'b-', linewidth=2, label='Best Fitness')
axes[2].plot(avg_history, 'c--', linewidth=1, label='Avg Fitness')
axes[2].set_xlabel('Generation', fontsize=12)
axes[2].set_ylabel('Fitness', fontsize=12)
axes[2].set_title('Learning Curve', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 수치 비교
print("=" * 50)
print("📊 학습 전 vs 학습 후 비교")
print("=" * 50)
print(f"  학습 전 평균 적합도: {np.mean(random_scores):5.1f}")
print(f"  학습 후 평균 적합도: {np.mean(trained_scores):5.1f}")
print(f"  향상 배율: {np.mean(trained_scores)/max(np.mean(random_scores), 0.1):.1f}x")
print()
print("💡 같은 신경망 구조(4→8→2)인데,")
print("   58개 가중치 '숫자'가 달라졌을 뿐입니다!")
print("   GA가 '좋은 숫자 조합'을 찾아준 것입니다.")


## 5단계: AI 학습 방법의 세계

GA 말고도 여러 학습 방법이 있습니다. 각각 **장단점**이 다릅니다.

### 주요 AI 학습 방법 비교

| 방법 | 원리 | 장점 | 단점 | 대표 응용 |
|------|------|------|------|----------|
| **경사 하강법** (Gradient Descent) | 손실 함수의 기울기를 따라 내려감 | 빠르고 정밀 | 미분 가능해야 함 | 이미지 인식, 번역 |
| **Hill Climbing** | 현재 해를 조금씩 변형 | 구현 간단, 빠른 수렴 | 지역 최적에 갇힐 수 있음 | 단순 최적화 |
| **유전 알고리즘** (GA) | 인구 + 선택 + 교차 + 변이 | 미분 불필요, 병렬 탐색 | 느린 수렴 | 로봇, 게임 AI |
| **강화학습** (RL) | 보상/벌을 통해 행동 학습 | 환경과 상호작용 학습 | 보상 설계 어려움 | AlphaGo, 로봇 |

### 자율주행 시뮬레이터에서 GA를 쓰는 이유

```
Q: 왜 경사 하강법(딥러닝)이 아니라 GA를 사용하나?

A: 자동차가 벽에 충돌하는 건 "미분할 수 없는" 이벤트입니다.
   경사 하강법은 "기울기"가 필요한데, 충돌 여부는 기울기가 없습니다.
   
   GA는 기울기가 필요 없습니다.
   "이 자동차가 얼마나 잘했나?" 점수만 있으면 됩니다.
   
   또한 GA는 여러 자동차를 동시에 돌릴 수 있어서
   시뮬레이터와 자연스럽게 어울립니다.
```


In [ ]:
# ============================================
# 🔬 실험: GA 하이퍼파라미터의 영향
# ============================================
# 인구 크기와 변이율이 학습에 어떤 영향을 미치는지 비교

configs = [
    {"pop": 10,  "mut_rate": 0.2, "mut_str": 0.3, "label": "Small Pop (10)"},
    {"pop": 30,  "mut_rate": 0.2, "mut_str": 0.3, "label": "Medium Pop (30)"},
    {"pop": 30,  "mut_rate": 0.05, "mut_str": 0.1, "label": "Low Mutation"},
    {"pop": 30,  "mut_rate": 0.4,  "mut_str": 0.5, "label": "High Mutation"},
]

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for cfg, color in zip(configs, colors):
    np.random.seed(42)
    pop = [Brain() for _ in range(cfg["pop"])]
    hist = []
    
    for gen in range(60):
        sc = [env.evaluate(b) for b in pop]
        hist.append(max(sc))
        
        sp = sorted(zip(sc, pop), key=lambda x: x[0], reverse=True)
        new_pop = [sp[i][1].copy() for i in range(max(2, cfg["pop"]//5))]
        while len(new_pop) < cfg["pop"]:
            ca = np.random.choice(cfg["pop"], 3, replace=False)
            cb = np.random.choice(cfg["pop"], 3, replace=False)
            pa = pop[ca[np.argmax([sc[c] for c in ca])]]
            pb = pop[cb[np.argmax([sc[c] for c in cb])]]
            new_pop.append(pa.crossover(pb).mutate(cfg["mut_rate"], cfg["mut_str"]))
        pop = new_pop
    
    ax.plot(hist, color=color, linewidth=2, label=f'{cfg["label"]} (final: {hist[-1]:.0f})')

ax.set_xlabel('Generation', fontsize=12)
ax.set_ylabel('Best Fitness', fontsize=12)
ax.set_title('Effect of GA Hyperparameters', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📊 관찰 포인트:")
print("  • 인구가 적으면(10) → 다양성 부족, 학습 불안정")
print("  • 변이가 너무 낮으면 → 정체될 수 있음")
print("  • 변이가 너무 높으면 → 좋은 해도 흩어짐")
print("  • 적절한 균형이 중요!")


## 📝 핵심 정리

### 오늘 확인한 것
1. **학습 전** = 랜덤 가중치 → 벽 앞에서 멈춤, 목표 도달 불가
2. **학습 후** = 최적화된 가중치 → 틈을 찾아 통과, 목표 도달!
3. 같은 신경망 구조인데 **숫자(가중치)만 달라졌을 뿐**
4. GA가 "시행착오"를 반복하며 좋은 숫자 조합을 찾아준 것

### 자율주행 시뮬레이터에서 볼 것
| 이 실습 | 시뮬레이터 |
|--------|----------|
| 2D 점 에이전트 | 삼각형 자동차 |
| 센서 4개 | 센서 N개 (설정 가능) |
| 벽 1개 + 틈 | 복잡한 트랙 |
| 적합도 1종류 | **보상 함수 5종류 선택** |
| 고정 파라미터 | **인구/변이율 조정 가능** |

> 이제 `simulator_v4.py`를 실행하여 실제 자동차의 학습을 관찰합니다!
> **실험 과제**: 같은 시드에서 보상 함수만 바꾸면 어떤 차이가 나는가?
